# Aula 11 — Laboratório de Boosting e Gradient Boosting

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joaopaulomirandamatias/ai-lab/blob/main/03-machine-learning/notebooks/11-gradient-boosting-laboratorio.ipynb)

Este laboratório constrói a intuição de correções residuais, valida o gradiente da perda quadrática e seleciona conjuntamente taxa de aprendizado e número de estágios sem consultar o teste.

**Protocolo:** separar teste → ajustar trajetórias em treino interno → selecionar em validação → reajustar no desenvolvimento → avaliar uma vez no teste.

## Ambiente e reprodutibilidade

Dependências mínimas:

```text
Python >= 3.10
numpy >= 1.24
pandas >= 2.0
matplotlib >= 3.7
scikit-learn >= 1.3
```

Os dados são sintéticos e gerados localmente. Não há download, segredo ou credencial.

In [ ]:
import os
import warnings

os.environ["MPLBACKEND"] = "Agg"
warnings.filterwarnings("error")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_friedman1
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

SEED = 20260908
rng = np.random.default_rng(SEED)

print({
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
})

## 1. Primeira correção residual

Para erro quadrático, o modelo inicial é a média e o pseudo-resíduo é $r_i=y_i-F(x_i)$. Reproduzimos primeiro o exemplo numérico da aula, sem biblioteca.

In [ ]:
y_toy = np.array([3.0, 5.0, 9.0])
f0 = np.full_like(y_toy, y_toy.mean())
residuos = y_toy - f0
correcao_toco = np.array([-2.0, -2.0, 3.0])
eta_toy = 0.1
f1 = f0 + eta_toy * correcao_toco

assert np.allclose(f0, 17 / 3)
assert np.allclose(residuos.sum(), 0.0)
assert np.allclose(f1, [5.4666666667, 5.4666666667, 5.9666666667])

pd.DataFrame({
    "y": y_toy,
    "F0": f0,
    "residuo": residuos,
    "h1": correcao_toco,
    "F1": f1,
}).round(3)

O toco aproxima, mas não copia perfeitamente, os resíduos. O `learning_rate` reduz a magnitude da atualização e deixa correções para os estágios seguintes.

## 2. Gradient check

Para $L(y,F)=	frac12(y-F)^2$, a derivada analítica é $F-y$ e o gradiente negativo é $y-F$. Uma diferença central verifica a derivação numericamente.

In [ ]:
def perda_quadratica(y, f):
    return 0.5 * (y - f) ** 2

y_check, f_check, eps = 12.0, 9.0, 1e-6
grad_numerico = (
    perda_quadratica(y_check, f_check + eps)
    - perda_quadratica(y_check, f_check - eps)
) / (2 * eps)
grad_analitico = f_check - y_check
erro_gradiente = abs(grad_numerico - grad_analitico)

assert erro_gradiente < 1e-8
print(f"gradiente analítico = {grad_analitico:.9f}")
print(f"gradiente numérico  = {grad_numerico:.9f}")
print(f"erro absoluto       = {erro_gradiente:.3e}")

## 3. Boosting manual com tocos

Agora cada `DecisionTreeRegressor(max_depth=1)` aprende os resíduos deixados pelo conjunto corrente. Não é uma reimplementação completa da biblioteca: é uma demonstração fiel do caso de erro quadrático com passo fixo.

In [ ]:
X_small = np.arange(8, dtype=float).reshape(-1, 1)
y_small = np.array([2.0, 2.8, 3.2, 5.4, 5.9, 8.3, 8.8, 9.1])
pred_small = np.full_like(y_small, y_small.mean())
mse_manual = [mean_squared_error(y_small, pred_small)]
componentes = []

for etapa in range(1, 11):
    residuo = y_small - pred_small
    stump = DecisionTreeRegressor(max_depth=1, random_state=SEED + etapa)
    stump.fit(X_small, residuo)
    atualizacao = stump.predict(X_small)
    pred_small += 0.2 * atualizacao
    componentes.append(stump)
    mse_manual.append(mean_squared_error(y_small, pred_small))

assert all(b < a for a, b in zip(mse_manual, mse_manual[1:]))
print(f"MSE inicial: {mse_manual[0]:.6f}")
print(f"MSE após 10 tocos: {mse_manual[-1]:.6f}")
print(f"redução relativa: {1 - mse_manual[-1] / mse_manual[0]:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(mse_manual)), mse_manual, marker="o")
ax.set(xlabel="Número de correções", ylabel="MSE de treino",
       title="Cada toco reduz o erro residual do conjunto")
ax.grid(alpha=0.25)
plt.show()

## 4. Dados, unidade de análise e partições

`make_friedman1` produz regressão não linear com interações. Cada linha é uma unidade independente. Guardamos índices explícitos para provar que teste, treino interno e validação não se sobrepõem.

In [ ]:
X, y = make_friedman1(
    n_samples=1800,
    n_features=10,
    noise=2.0,
    random_state=SEED,
)
ids = np.arange(len(y))

X_dev, X_test, y_dev, y_test, id_dev, id_test = train_test_split(
    X, y, ids, test_size=0.25, random_state=SEED
)
X_train, X_val, y_train, y_val, id_train, id_val = train_test_split(
    X_dev, y_dev, id_dev, test_size=0.25, random_state=SEED
)

assert X.shape == (1800, 10)
assert set(id_train).isdisjoint(id_val)
assert set(id_train).isdisjoint(id_test)
assert set(id_val).isdisjoint(id_test)
assert len(id_train) + len(id_val) + len(id_test) == len(ids)

print({"treino_interno": len(y_train), "validacao": len(y_val), "teste": len(y_test)})

## 5. Baselines

Comparamos a média e uma árvore rasa. O teste continua fechado; estas métricas são de validação.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
arvore = DecisionTreeRegressor(
    max_depth=3, min_samples_leaf=10, random_state=SEED
).fit(X_train, y_train)

rmse_base_val = rmse(y_val, baseline.predict(X_val))
rmse_arvore_val = rmse(y_val, arvore.predict(X_val))

assert rmse_arvore_val < rmse_base_val
print(f"RMSE validação — média:  {rmse_base_val:.6f}")
print(f"RMSE validação — árvore: {rmse_arvore_val:.6f}")

## 6. Trajetórias por taxa de aprendizado

Treinamos um orçamento de 400 estágios para cada taxa. `staged_predict` mede o RMSE após 1, 2, ..., 400 árvores. A escolha usa somente validação.

In [ ]:
taxas = [0.02, 0.05, 0.10, 0.20]
orcamento = 400
trajetorias = {}
linhas = []

for taxa in taxas:
    modelo = GradientBoostingRegressor(
        loss="squared_error",
        learning_rate=taxa,
        n_estimators=orcamento,
        max_depth=2,
        min_samples_leaf=8,
        subsample=1.0,
        random_state=SEED,
    ).fit(X_train, y_train)
    rmse_treino = np.array([rmse(y_train, p) for p in modelo.staged_predict(X_train)])
    rmse_val = np.array([rmse(y_val, p) for p in modelo.staged_predict(X_val)])
    melhor_indice = int(np.argmin(rmse_val))
    trajetorias[taxa] = (rmse_treino, rmse_val)
    linhas.append({
        "learning_rate": taxa,
        "melhor_estagio": melhor_indice + 1,
        "rmse_validacao": rmse_val[melhor_indice],
        "rmse_treino_no_estagio": rmse_treino[melhor_indice],
    })

selecao = pd.DataFrame(linhas).sort_values("rmse_validacao").reset_index(drop=True)
melhor_taxa = float(selecao.loc[0, "learning_rate"])
melhor_estagio = int(selecao.loc[0, "melhor_estagio"])

assert 1 <= melhor_estagio <= orcamento
assert selecao["rmse_validacao"].notna().all()
selecao.round(6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for taxa, (tr, va) in trajetorias.items():
    axes[0].plot(np.arange(1, orcamento + 1), va, label=f"eta={taxa}")
axes[0].axvline(melhor_estagio, color="black", linestyle="--", alpha=0.6)
axes[0].set(xlabel="Estágio", ylabel="RMSE de validação",
            title="Taxa e número de árvores são acoplados")
axes[0].legend()
axes[0].grid(alpha=0.25)

tr_best, va_best = trajetorias[melhor_taxa]
axes[1].plot(tr_best, label="treino")
axes[1].plot(va_best, label="validação")
axes[1].axvline(melhor_estagio - 1, color="black", linestyle="--")
axes[1].set(xlabel="Estágio (índice iniciado em zero)", ylabel="RMSE",
            title=f"Trajetória escolhida: eta={melhor_taxa}")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

A perda de treino tende a continuar caindo. A validação define o compromisso relevante. Se o mínimo cair no limite do orçamento, isso significa apenas que a busca deveria testar mais estágios — não autoriza olhar o teste.

## 7. Reajuste e abertura única do teste

Congelamos a configuração escolhida, reajustamos no desenvolvimento completo e só então avaliamos o teste.

In [ ]:
modelo_final = GradientBoostingRegressor(
    loss="squared_error",
    learning_rate=melhor_taxa,
    n_estimators=melhor_estagio,
    max_depth=2,
    min_samples_leaf=8,
    subsample=1.0,
    random_state=SEED,
).fit(X_dev, y_dev)

baseline_final = DummyRegressor(strategy="mean").fit(X_dev, y_dev)
arvore_final = DecisionTreeRegressor(
    max_depth=3, min_samples_leaf=10, random_state=SEED
).fit(X_dev, y_dev)

pred_final = modelo_final.predict(X_test)
resultados_teste = pd.DataFrame([
    {"modelo": "Média", "RMSE": rmse(y_test, baseline_final.predict(X_test)),
     "MAE": mean_absolute_error(y_test, baseline_final.predict(X_test)),
     "R2": r2_score(y_test, baseline_final.predict(X_test))},
    {"modelo": "Árvore rasa", "RMSE": rmse(y_test, arvore_final.predict(X_test)),
     "MAE": mean_absolute_error(y_test, arvore_final.predict(X_test)),
     "R2": r2_score(y_test, arvore_final.predict(X_test))},
    {"modelo": "Gradient Boosting", "RMSE": rmse(y_test, pred_final),
     "MAE": mean_absolute_error(y_test, pred_final),
     "R2": r2_score(y_test, pred_final)},
])

assert resultados_teste.loc[2, "RMSE"] < resultados_teste.loc[0, "RMSE"]
assert resultados_teste.loc[2, "RMSE"] < resultados_teste.loc[1, "RMSE"]
print(f"configuração congelada: eta={melhor_taxa}, estágios={melhor_estagio}")
resultados_teste.round(6)

## 8. Resíduos no teste

O gráfico é diagnóstico posterior, não uma nova rodada de tuning. Padrões estruturados indicariam regiões em que o modelo ainda erra sistematicamente.

In [ ]:
residuos_teste = y_test - pred_final
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(pred_final, residuos_teste, s=16, alpha=0.55)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(xlabel="Predição", ylabel="Resíduo (y - predição)",
            title="Resíduos versus predições")
axes[1].hist(residuos_teste, bins=24, edgecolor="white")
axes[1].set(xlabel="Resíduo", ylabel="Frequência", title="Distribuição dos resíduos")
plt.tight_layout()
plt.show()

print(f"resíduo médio: {residuos_teste.mean():.6f}")
print(f"resíduo absoluto mediano: {np.median(np.abs(residuos_teste)):.6f}")

## 9. Boosting estocástico e estabilidade entre seeds

`subsample=0.7` faz cada estágio usar uma subamostra. Como essa variante é aleatória, medimos sua dispersão em oito seeds no mesmo conjunto de validação. Não usamos esses resultados para reabrir a escolha final.

In [ ]:
linhas_seed = []
for fracao in [1.0, 0.7]:
    for deslocamento in range(8):
        modelo = GradientBoostingRegressor(
            loss="squared_error",
            learning_rate=melhor_taxa,
            n_estimators=melhor_estagio,
            max_depth=2,
            min_samples_leaf=8,
            subsample=fracao,
            random_state=SEED + deslocamento,
        ).fit(X_train, y_train)
        linhas_seed.append({
            "subsample": fracao,
            "seed": SEED + deslocamento,
            "rmse_validacao": rmse(y_val, modelo.predict(X_val)),
        })

estabilidade = pd.DataFrame(linhas_seed)
resumo_estabilidade = estabilidade.groupby("subsample")["rmse_validacao"].agg(
    ["mean", "std", "min", "max"]
)

assert len(estabilidade) == 16
assert np.isfinite(estabilidade["rmse_validacao"]).all()
resumo_estabilidade.round(6)

Com `subsample=1`, mudar `random_state` não altera este estimador determinístico de forma material. Com subamostragem, a dispersão torna-se parte do resultado e deve ser registrada.

## 10. Sanidade para classificação

Na log-loss binária, o pseudo-resíduo é $y-p$. O cálculo abaixo confirma o sinal das duas correções.

In [ ]:
y_cls = np.array([1.0, 0.0])
p_cls = np.array([0.8, 0.3])
pseudo_residuos_cls = y_cls - p_cls
assert np.allclose(pseudo_residuos_cls, [0.2, -0.3])
print("pseudo-resíduos binários:", pseudo_residuos_cls.tolist())

## 11. Verificações finais

In [ ]:
assert not np.isnan(X).any()
assert not np.isnan(y).any()
assert len(modelo_final.estimators_) == melhor_estagio
assert modelo_final.n_features_in_ == X.shape[1]
assert np.isfinite(pred_final).all()
assert np.isfinite(residuos_teste).all()

print("Todas as verificações passaram.")
print(f"Amostras: {len(y)} | atributos: {X.shape[1]} | seed: {SEED}")
print(f"RMSE final: {rmse(y_test, pred_final):.6f} | R² final: {r2_score(y_test, pred_final):.6f}")

## Conclusões

- O caso quadrático transforma gradient boosting em correções sucessivas de resíduos.
- `learning_rate` e número de estágios precisam ser escolhidos em conjunto.
- A validação escolheu a configuração; o teste permaneceu isolado até o reajuste final.
- O ensemble superou a média e a árvore rasa neste conjunto sintético.
- A subamostragem introduziu variabilidade mensurável entre seeds.

O próximo passo curricular é comparar essa construção aditiva de regiões com a geometria de margens e kernels das Support Vector Machines.